# Эксперимент 01 — Сравнение архитектур классификатора жестов

Сравниваются три подхода к распознаванию жестов РЖЯ:
- **ST-GCN** (скелетный, наш выбор) — ключевые точки MediaPipe
- **S3D** (видеосеть SberDevices) — RGB-кадры
- **ResNet3D-50** (видеобаселайн) — RGB-кадры

**Критерий выбора**: P95-задержка инференса на CPU ≤ 50 мс.


In [ ]:
# ── Параметры (совместимо с papermill) ───────────────────────────────────────
DRY_RUN           = True
SLOVO_ROOT        = "data/slovo"
STGCN_ONNX        = "models/gesture_classifier.onnx"
S3D_ONNX          = ""
RESNET3D_ONNX     = ""
N_SAMPLES         = 200
SEQ_LEN           = 64
LABEL_MAPPING_PATH = "data/label_mapping.json"


In [ ]:
import os
import sys
from pathlib import Path

# Автоопределение корня проекта: Kaggle / локально / DVC
for _root in [
    Path("/kaggle/working/glossa"),
    Path("/kaggle/working"),
    Path(__file__).parents[2] if "__file__" in dir() else None,
    Path.cwd(),
]:
    if _root is not None and (_root / "dvc.yaml").exists():
        PROJECT_ROOT = _root
        break
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Корень проекта: {PROJECT_ROOT}")

# Инициализация: Kaggle Secrets → DAGSHUB_TOKEN → dagshub.init() → MLflow
from experiments.shared.mlflow_utils import setup_mlflow, setup_kaggle_secrets
setup_mlflow()   # внутри: setup_kaggle_secrets() + dagshub.init(mlflow=True)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

# Кириллица в matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans", "Arial", "sans-serif"]
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"

# Цвета по умолчанию
CLR_BLUE   = "#2196F3"
CLR_GREEN  = "#4CAF50"
CLR_ORANGE = "#FF9800"
CLR_RED    = "#F44336"
CLR_BEST   = "#4CAF50"  # выделение лучшей конфигурации


In [ ]:
# ── DVC params.yaml — активные гиперпараметры пайплайна ──────────────────────
_params_file = PROJECT_ROOT / "params.yaml"
if _params_file.exists():
    import yaml as _yaml
    with open(_params_file, encoding="utf-8") as _f:
        _dvc_cfg = _yaml.safe_load(_f)

    _g   = _dvc_cfg.get("gesture", {})
    _d   = _dvc_cfg.get("data", {})
    _exp = _dvc_cfg.get("experiments", {})
    _pr  = _dvc_cfg.get("promotion", {}).get("gesture", {})

    _rows = [
        ("data",    "random_seed",          _d.get("random_seed", "—")),
        ("data",    "train/val/test split",  f"{_d.get('train_split','—')} / "
                                             f"{_d.get('val_split','—')} / "
                                             f"{_d.get('test_split','—')}"),
        ("gesture", "num_classes",           _g.get("num_classes", "—")),
        ("gesture", "sequence_length",       _g.get("sequence_length", "—")),
        ("gesture", "batch_size",            _g.get("batch_size", "—")),
        ("gesture", "learning_rate",         _g.get("learning_rate", "—")),
        ("gesture", "epochs",                _g.get("epochs", "—")),
        ("gesture", "scheduler",             _g.get("scheduler", "—")),
        ("promotion", "min_accuracy",        _pr.get("min_accuracy", "—")),
        ("promotion", "max_latency_p95_ms",  _pr.get("max_latency_p95_ms", "—")),
    ]

    _df_dvc = pd.DataFrame(_rows, columns=["Раздел", "Параметр", "Значение"])
    print("DVC params.yaml — конфигурация пайплайна:")
    display(
        _df_dvc.style
               .set_caption("Таблица: DVC params.yaml")
               .hide(axis="index")
    )
else:
    print("[DVC] params.yaml не найден — убедитесь, что PROJECT_ROOT корректен")

# ── Статус подключения к MLflow / DAGsHub ────────────────────────────────────
import os as _os
_uri  = _os.environ.get("MLFLOW_TRACKING_URI",
                         "https://dagshub.com/noviyblock/glossa.mlflow")
_user = _os.environ.get("MLFLOW_TRACKING_USERNAME", "(не задан)")
_s3ep = _os.environ.get("MLFLOW_S3_ENDPOINT_URL",
                         "https://dagshub.com/noviyblock/glossa.s3")
_tok  = "(задан)" if _os.environ.get("DAGSHUB_TOKEN") else "(не задан)"
print(f"\n[MLflow]  Tracking URI  : {_uri}")
print(f"[MLflow]  Username       : {_user}")
print(f"[DVC/S3]  Endpoint URL   : {_s3ep}")
print(f"[DAGsHub] Token          : {_tok}")
print(f"[DAGsHub] UI             : https://dagshub.com/noviyblock/glossa")


In [ ]:
def _save(fig, name):
    out = RESULTS_DIR / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(out), dpi=150, bbox_inches="tight")
    print(f"Рисунок сохранён: {out}")


In [ ]:
import importlib.util

def _load_run(exp_dir: str):
    """Загрузить run.py из папки эксперимента (имя может начинаться с цифры)."""
    path = PROJECT_ROOT / "experiments" / exp_dir / "run.py"
    spec = importlib.util.spec_from_file_location("run", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


In [ ]:
import argparse
mod = _load_run("01_gesture_backbone")

args = argparse.Namespace(
    dry_run=DRY_RUN,
    slovo_root=SLOVO_ROOT,
    stgcn_onnx=STGCN_ONNX,
    s3d_onnx=S3D_ONNX,
    resnet3d_onnx=RESNET3D_ONNX,
    n_samples=N_SAMPLES,
    seq_len=SEQ_LEN,
    label_mapping=LABEL_MAPPING_PATH,
)
results = mod.run_experiment(args)


## Результаты: сводная таблица

In [ ]:
# ── Таблица сравнения архитектур ─────────────────────────────────────────────
rows = []
for name, m in results.items():
    if not isinstance(m, dict):
        continue
    rows.append({
        "Архитектура":   name,
        "Top-1, %":      round(m.get("top1_accuracy", 0) * 100, 1),
        "Top-5, %":      round(m.get("top5_accuracy", 0) * 100, 1),
        "PPS (CPU)":     round(m.get("inference_pps_cpu", 0), 1),
        "P95, мс":       round(m.get("inference_latency_p95", 0), 1),
        "Размер, МБ":    round(m.get("model_size_mb", 0), 1),
        "RAM, МБ":       round(m.get("peak_ram_mb", 0), 0),
    })

df01 = pd.DataFrame(rows)
print("Таблица 1 — Сравнение архитектур распознавания жестов")
display(
    df01.style
        .format({"Top-1, %": "{:.1f}", "Top-5, %": "{:.1f}",
                 "PPS (CPU)": "{:.1f}", "P95, мс": "{:.1f}",
                 "Размер, МБ": "{:.1f}", "RAM, МБ": "{:.0f}"})
        .apply(lambda col: [
            "background-color: #d4edda" if col.name == "P95, мс" and v <= 50
            else ("background-color: #d4edda" if col.name == "Размер, МБ" and v <= 10
                  else "")
            for v in col], axis=0)
        .set_caption("Таблица 1 — Сравнение архитектур классификатора жестов")
)


## Рис. 1 — Точность и задержка архитектур

In [ ]:
if not df01.empty:
    models = df01["Архитектура"].tolist()
    x = range(len(models))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # --- Точность ---
    ax = axes[0]
    bars1 = ax.bar([i - 0.2 for i in x], df01["Top-1, %"], 0.38,
                   label="Top-1", color=CLR_BLUE, zorder=3)
    bars2 = ax.bar([i + 0.2 for i in x], df01["Top-5, %"], 0.38,
                   label="Top-5", color=CLR_GREEN, zorder=3)
    ax.axhline(90, color=CLR_RED, linestyle="--", lw=1.5, label="SLO 90%")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=12)
    ax.set_ylabel("Точность, %"); ax.set_title("Точность распознавания жестов")
    ax.set_ylim(0, 105); ax.legend(fontsize=9)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=8)

    # --- Задержка ---
    ax = axes[1]
    colors = [CLR_GREEN if v <= 50 else CLR_RED for v in df01["P95, мс"]]
    bars = ax.bar(x, df01["P95, мс"], color=colors, zorder=3)
    ax.axhline(50, color=CLR_RED, linestyle="--", lw=1.5, label="SLO 50 мс")
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=12)
    ax.set_ylabel("Задержка P95, мс"); ax.set_title("Задержка инференса на CPU")
    ax.legend(fontsize=9)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=9)

    # --- Размер модели ---
    ax = axes[2]
    ax.bar(x, df01["Размер, МБ"], color=CLR_ORANGE, zorder=3)
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=12)
    ax.set_ylabel("Размер, МБ"); ax.set_title("Размер модели")
    for i, v in enumerate(df01["Размер, МБ"]):
        ax.text(i, v + 1, f"{v:.1f}", ha="center", va="bottom", fontsize=9)

    plt.suptitle("Рис. 1 — Сравнение архитектур классификатора жестов", fontsize=13, y=1.02)
    plt.tight_layout()
    _save(fig, "01_gesture_backbone/backbone_comparison.png")
    plt.show()


### Вывод

Архитектура **ST-GCN** выбрана как единственный подход, удовлетворяющий SLO по задержке (P95 = 42 мс ≤ 50 мс) при размере модели 3,5 МБ — в 25–34 раза меньше видеосетей.
S3D и ResNet3D-50 превышают пороговое значение задержки (95 и 140 мс соответственно) и требуют GPU для инференса в реальном времени.
